In [8]:
import pandas as pd
import os
import glob

# ==========================================
# 1. KONFIGURASI FILE & FOLDER
# ==========================================
INPUT_DIR = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Dataset/RAW'  
OUTPUT_DIR = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Dataset/PerPantai'  
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mengambil semua file yang berakhiran _TA_RAW.csv
file_pattern = os.path.join(INPUT_DIR, '*_TA_RAW.csv')
ta_files = glob.glob(file_pattern)

if not ta_files:
    print("❌ Tidak ada file _TA_RAW.csv yang ditemukan di direktori!")
    exit()

print(f"✅ Ditemukan {len(ta_files)} file TripAdvisor RAW.\n")

# Dictionary untuk menyimpan persentase null per file
null_percentages = {}
all_columns_set = set()

# ==========================================
# 2. BACA DATA & HITUNG MISSING VALUES
# ==========================================
for file_path in ta_files:
    # Mempercantik nama kolom header (misal: "Kuta" dari "Kuta_TA_RAW.csv")
    filename = os.path.basename(file_path)
    beach_name = filename.split('_')[0] 
    
    # Membaca seluruh baris untuk menghitung null
    df = pd.read_csv(file_path)
    
    # Update daftar total kolom unik dari semua file
    all_columns_set.update(df.columns)
    
    total_rows = len(df)
    # Menghitung persentase null
    null_pct = (df.isnull().sum() / total_rows * 100).round(2)
    
    # Menyimpan informasi persentase ke dalam dictionary berdasarkan nama pantai
    null_percentages[beach_name] = null_pct

# ==========================================
# 3. MEMBUAT TABEL GABUNGAN
# ==========================================
# Membuat DataFrame dari dictionary. Index-nya adalah nama kolom asli.
df_summary = pd.DataFrame(null_percentages)

# Jika ada kolom yang tidak ada di salah satu pantai, pandas akan mengisi NaN.
# Kita ganti NaN menjadi 100 (karena berarti 100% data di kolom itu hilang/tidak ada untuk pantai tersebut).
df_summary = df_summary.fillna(100.0)

# Menambahkan kolom "Rata-rata Null (%)" untuk mengurutkan tabel dari null tertinggi ke terendah secara global
df_summary['Avg_Null(%)'] = df_summary.mean(axis=1).round(2)
df_summary = df_summary.sort_values(by='Avg_Null(%)', ascending=False)

# Memformat output agar lebih rapi (menambahkan simbol % di tampilan print)
df_print = df_summary.copy()
for col in df_print.columns:
    df_print[col] = df_print[col].apply(lambda x: f"{x:.2f}%")

# ==========================================
# 4. TAMPILKAN HASIL TABEL
# ==========================================
print("="*100)
print("TABEL PERSENTASE MISSING VALUES (NULL) PER PANTAI")
print("="*100)

# Mengatur pandas agar saat diprint di terminal tidak terpotong ke bawah/menyamping
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(df_print.to_string())
print("="*100)

✅ Ditemukan 7 file TripAdvisor RAW.

TABEL PERSENTASE MISSING VALUES (NULL) PER PANTAI
                                      Seminyak PadangPadang    Sanur Kelingking  Pandawa     Kuta  NusaDua Avg_Null(%)
user                                   100.00%      100.00%  100.00%    100.00%  100.00%  100.00%  100.00%     100.00%
user/avatar                            100.00%      100.00%  100.00%    100.00%  100.00%  100.00%  100.00%     100.00%
placeInfo/website                      100.00%      100.00%  100.00%    100.00%  100.00%  100.00%  100.00%     100.00%
publishedPlatform                      100.00%      100.00%  100.00%    100.00%  100.00%  100.00%  100.00%     100.00%
user/userLocation                      100.00%      100.00%  100.00%    100.00%  100.00%  100.00%  100.00%     100.00%
ownerResponse                          100.00%      100.00%  100.00%    100.00%  100.00%  100.00%  100.00%     100.00%
placeInfo/addressObj/street2           100.00%      100.00%  100.00%    100.00% 

In [9]:
import pandas as pd
import os
import glob

# ==========================================
# 1. KONFIGURASI FOLDER & KOLOM TARGET
# ==========================================
INPUT_DIR = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Dataset/RAW'
OUTPUT_DIR = '/Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Dataset/PerPantai'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Daftar kolom yang ingin dipertahankan
TARGET_COLUMNS_TA = [
    # --- METADATA DESTINASI ---
    'placeInfo/name',
    
    # --- IDENTITAS REVIEW & URL ---
    'id',               # ID Ulasan (Tambahan krusial untuk deduplikasi)
    'url',
    
    # --- IDENTITAS PENGGUNA ---
    'user/userId',
    'user/name',
    'user/userLocation/name',
    
    # --- KONTEKS PERJALANAN & WAKTU ---
    'tripType',
    'travelDate',
    'publishedDate',
    
    # --- ISI ULASAN (INTI NLP) ---
    'rating',           # Rating bintang yang diberikan reviewer
    'text',             # Isi ulasan
    'helpfulVotes'      # Jumlah vote berguna (Tambahan opsional yang bagus)
]

# Mencari semua file _TA_RAW.csv
file_pattern = os.path.join(INPUT_DIR, '*_TA_RAW.csv')
ta_files = glob.glob(file_pattern)

if not ta_files:
    print("❌ Tidak ada file _TA_RAW.csv yang ditemukan di folder RAW!")
    exit()

print(f"✅ Ditemukan {len(ta_files)} file TripAdvisor RAW.\n")

# ==========================================
# 2. VALIDASI KOLOM (CEK KETERSEDIAAN)
# ==========================================
print("="*70)
print("TAHAP 1: PENGECEKAN KOLOM TARGET DI SETIAP FILE")
print("="*70)

all_valid = True
for file_path in ta_files:
    filename = os.path.basename(file_path)
    # Baca header saja untuk validasi cepat
    df_temp = pd.read_csv(file_path, nrows=0)
    existing_cols = set(df_temp.columns)
    
    missing_cols = [col for col in TARGET_COLUMNS_TA if col not in existing_cols]
    
    if missing_cols:
        all_valid = False
        print(f"⚠️ {filename} -> KURANG KOLOM: {missing_cols}")
    else:
        print(f"✅ {filename} -> Lengkap (11 kolom tersedia)")

print("-" * 70)
if all_valid:
    print("🎉 SEMUA FILE MEMILIKI KOLOM YANG LENGKAP!")
else:
    print("⚠️ Ada file yang kehilangan beberapa kolom target.")
    print("   Sistem akan otomatis menambahkan kolom tersebut dengan isi kosong (NaN) agar format seragam.")

# ==========================================
# 3. EKSTRAKSI DAN PENYIMPANAN KE FILE BARU
# ==========================================
print("\n" + "="*70)
print(f"TAHAP 2: EKSTRAKSI & SIMPAN KE FOLDER {OUTPUT_DIR}")
print("="*70)

for file_path in ta_files:
    filename = os.path.basename(file_path)
    # Hapus '_RAW' dari nama file (misal: Kuta_TA_RAW.csv -> Kuta_TA.csv)
    new_filename = filename.replace('_RAW.csv', '.csv')
    output_path = os.path.join(OUTPUT_DIR, new_filename)
    
    # Baca seluruh data
    df = pd.read_csv(file_path)
    
    # Jika ada kolom target yang tidak ada di dataframe, buat kolomnya dan isi dengan NaN
    for col in TARGET_COLUMNS_TA:
        if col not in df.columns:
            df[col] = pd.NA
            
    # Filter hanya kolom target dan urutkan sesuai list TARGET_COLUMNS_TA
    df_filtered = df[TARGET_COLUMNS_TA].copy()
    
    # Simpan ke folder PerPantai
    df_filtered.to_csv(output_path, index=False)
    
    print(f"💾 Tersimpan: {new_filename} ({len(df_filtered)} baris, {len(df_filtered.columns)} kolom)")

print("\n🚀 PROSES SELESAI! Seluruh file siap digunakan untuk tahap Preprocessing NLP.")

✅ Ditemukan 7 file TripAdvisor RAW.

TAHAP 1: PENGECEKAN KOLOM TARGET DI SETIAP FILE
✅ Seminyak_TA_RAW.csv -> Lengkap (11 kolom tersedia)
✅ PadangPadang_TA_RAW.csv -> Lengkap (11 kolom tersedia)
✅ Sanur_TA_RAW.csv -> Lengkap (11 kolom tersedia)
✅ Kelingking_TA_RAW.csv -> Lengkap (11 kolom tersedia)
✅ Pandawa_TA_RAW.csv -> Lengkap (11 kolom tersedia)
✅ Kuta_TA_RAW.csv -> Lengkap (11 kolom tersedia)
✅ NusaDua_TA_RAW.csv -> Lengkap (11 kolom tersedia)
----------------------------------------------------------------------
🎉 SEMUA FILE MEMILIKI KOLOM YANG LENGKAP!

TAHAP 2: EKSTRAKSI & SIMPAN KE FOLDER /Users/natasyays/Downloads/Documents/SEMPRO/bali dataset/Dataset/PerPantai
💾 Tersimpan: Seminyak_TA.csv (999 baris, 12 kolom)
💾 Tersimpan: PadangPadang_TA.csv (999 baris, 12 kolom)
💾 Tersimpan: Sanur_TA.csv (999 baris, 12 kolom)
💾 Tersimpan: Kelingking_TA.csv (997 baris, 12 kolom)
💾 Tersimpan: Pandawa_TA.csv (999 baris, 12 kolom)
💾 Tersimpan: Kuta_TA.csv (999 baris, 12 kolom)
💾 Tersimpan: Nus